In [ ]:
!pip install scikit-fuzzy

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt

# ---------- Universos de discurso ----------
# Sintomas: 0 = ningun sintoma respiratorio, 10 = maxima intensidad/cantidad
sintomas = ctrl.Antecedent(np.arange(0, 11, 1), 'sintomas')
# Edad: 0 a 100 anios
edad = ctrl.Antecedent(np.arange(0, 101, 1), 'edad')
# Duracion de los sintomas: 0 a 15 dias
duracion = ctrl.Antecedent(np.arange(0, 16, 1), 'duracion')
# Salida: nivel de enfermedad, 0 = leve, 10 = grave
enfermedad = ctrl.Consequent(np.arange(0, 11, 1), 'enfermedad')

# ---------- Funciones de membresia (entrada: sintomas) ----------
sintomas['pocos'] = fuzz.trimf(sintomas.universe, [0, 0, 4])
sintomas['medios'] = fuzz.trimf(sintomas.universe, [2, 5, 8])
sintomas['altos'] = fuzz.trimf(sintomas.universe, [6, 10, 10])

# ---------- Funciones de membresia (entrada: edad) ----------
edad['baja'] = fuzz.trimf(edad.universe, [0, 0, 35])
edad['media'] = fuzz.trimf(edad.universe, [20, 45, 70])
edad['alta'] = fuzz.trimf(edad.universe, [55, 100, 100])

# ---------- Funciones de membresia (entrada: duracion de sintomas) ----------
duracion['reciente'] = fuzz.trimf(duracion.universe, [0, 0, 3])
duracion['persistente'] = fuzz.trimf(duracion.universe, [2, 6, 9])
duracion['prolongada'] = fuzz.trimf(duracion.universe, [7, 15, 15])

# ---------- Funciones de membresia (salida: enfermedad) ----------
enfermedad['poca'] = fuzz.trimf(enfermedad.universe, [0, 0, 4])
enfermedad['media'] = fuzz.trimf(enfermedad.universe, [2, 5, 8])
enfermedad['alta'] = fuzz.trimf(enfermedad.universe, [6, 10, 10])

# ---------- Reglas (9 en total, minimo pedido: 5) ----------
r1 = ctrl.Rule(sintomas['pocos'] & edad['baja'] & duracion['reciente'], enfermedad['poca'])
r2 = ctrl.Rule(sintomas['pocos'] & edad['media'] & duracion['reciente'], enfermedad['poca'])
r3 = ctrl.Rule(sintomas['pocos'] & edad['alta'], enfermedad['media'])
r4 = ctrl.Rule(sintomas['medios'] & edad['baja'] & duracion['reciente'], enfermedad['media'])
r5 = ctrl.Rule(sintomas['medios'] & edad['media'], enfermedad['media'])
r6 = ctrl.Rule(sintomas['medios'] & edad['alta'], enfermedad['alta'])
r7 = ctrl.Rule(sintomas['altos'], enfermedad['alta'])
r8 = ctrl.Rule(duracion['prolongada'], enfermedad['alta'])
r9 = ctrl.Rule(sintomas['pocos'] & duracion['prolongada'], enfermedad['media'])

sistema_ctrl = ctrl.ControlSystem([r1, r2, r3, r4, r5, r6, r7, r8, r9])

def diagnosticar_difuso(nivel_sintomas, edad_paciente, dias_duracion):
    """
    nivel_sintomas: 0-10 (intensidad/cantidad de sintomas respiratorios)
    edad_paciente: 0-100
    dias_duracion: 0-15 (dias con los sintomas)
    Devuelve: (valor_crisp, etiqueta, grados_pertenencia_dict)
    """
    sim = ctrl.ControlSystemSimulation(sistema_ctrl)
    sim.input['sintomas'] = nivel_sintomas
    sim.input['edad'] = edad_paciente
    sim.input['duracion'] = dias_duracion
    sim.compute()
    valor = sim.output['enfermedad']

    # Grado de pertenencia del resultado a cada etiqueta de salida
    grados = {
        'poca': fuzz.interp_membership(enfermedad.universe, enfermedad['poca'].mf, valor),
        'media': fuzz.interp_membership(enfermedad.universe, enfermedad['media'].mf, valor),
        'alta': fuzz.interp_membership(enfermedad.universe, enfermedad['alta'].mf, valor),
    }
    etiqueta = max(grados, key=grados.get)
    return valor, etiqueta, grados

# Prueba rapida
print(diagnosticar_difuso(7, 65, 5))

: 

In [ ]:
fig, axs = plt.subplots(4, 1, figsize=(7, 10))
sintomas.view(ax=axs[0])
axs[0].set_title("Sintomas respiratorios")
edad.view(ax=axs[1])
axs[1].set_title("Edad")
duracion.view(ax=axs[2])
axs[2].set_title("Duracion de sintomas (dias)")
enfermedad.view(ax=axs[3])
axs[3].set_title("Nivel de enfermedad (salida)")
plt.tight_layout()
plt.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

estilos_df = """
<style>
.df-card {
    background: linear-gradient(135deg, #fbf8ff, #f1eafb);
    border: 1px solid #e0d6f5;
    border-radius: 18px;
    padding: 24px 28px;
    font-family: 'Segoe UI', Arial, sans-serif;
    max-width: 620px;
    box-shadow: 0 8px 24px rgba(60, 20, 90, 0.12);
}
.df-title h3 { margin: 0 0 4px 0; color: #4a1f88; font-size: 22px; }
.df-title p { margin: 0 0 10px 0; color: #2c3440; font-size: 13px; }

.df-card .widget-label,
.df-card .widget-readout,
.df-card label {
    color: #2c3440 !important;
    font-weight: 600 !important;
    opacity: 1 !important;
}
.df-card .widget-slider,
.df-card .widget-hslider {
    background-color: transparent !important;
}

.df-card .df-btn button.widget-button,
.df-card .df-btn button {
    background-color: #6a1fb8 !important;
    color: #fff !important;
    font-weight: 700 !important;
    border: none !important;
    border-radius: 10px !important;
    padding: 9px 24px !important;
    margin-top: 10px !important;
    box-shadow: 0 3px 8px rgba(106, 31, 184, 0.35) !important;
}
.df-card .df-btn button:hover { background-color: #57188f !important; }

.df-result {
    border-radius: 12px;
    padding: 14px 18px;
    margin-top: 14px;
}
.df-poca { background: #e6f4ea !important; border-left: 5px solid #2e7d32; color: #0d2419; }
.df-media { background: #fff8e1 !important; border-left: 5px solid #f9a825; color: #3d2c00; }
.df-alta { background: #fdecea !important; border-left: 5px solid #c62828; color: #4a0e0e; }
.df-result-title { font-size: 16px; font-weight: 700; }
.df-result-body { font-size: 13px; margin-top: 6px; line-height: 1.4; }
</style>
"""
display(HTML(estilos_df))

slider_sintomas = widgets.FloatSlider(value=3, min=0, max=10, step=0.5,
    description='Sintomas:', continuous_update=False)
slider_edad = widgets.FloatSlider(value=30, min=0, max=100, step=1,
    description='Edad:', continuous_update=False)
slider_duracion = widgets.FloatSlider(value=2, min=0, max=15, step=1,
    description='Dias:', continuous_update=False)

boton_df = widgets.Button(description="Calcular", icon="calculator")
boton_df.add_class("df-btn")

salida_df = widgets.Output()

recomendaciones_df = {
    'poca': "Riesgo bajo. Sintomas leves, recientes y/o edad favorable. Se recomienda observacion en casa.",
    'media': "Riesgo moderado. Vigilar evolucion; considerar consulta si los sintomas aumentan o persisten.",
    'alta': "Riesgo alto. Se recomienda atencion medica pronta, especialmente si hay dificultad para respirar o los sintomas ya llevan varios dias.",
}

def al_calcular(b):
    with salida_df:
        clear_output()
        valor, etiqueta, grados = diagnosticar_difuso(
            slider_sintomas.value, slider_edad.value, slider_duracion.value
        )
        html = f'''
        <div class="df-result df-{etiqueta}">
            <div class="df-result-title">Nivel de enfermedad: {etiqueta.upper()} ({valor:.2f}/10)</div>
            <div class="df-result-body">{recomendaciones_df[etiqueta]}</div>
            <div class="df-result-body">Pertenencia -> poca: {grados['poca']:.2f} | media: {grados['media']:.2f} | alta: {grados['alta']:.2f}</div>
        </div>
        '''
        display(HTML(html))

boton_df.on_click(al_calcular)

titulo_df = widgets.HTML(
    "<div class='df-title'><h3>Sistema Difuso - Riesgo Respiratorio</h3>"
    "<p>Ajusta el nivel de sintomas, la edad y los dias de duracion, luego presiona 'Calcular'.</p></div>"
)

contenedor_df = widgets.VBox(
    [titulo_df, slider_sintomas, slider_edad, slider_duracion, boton_df, salida_df]
)
contenedor_df.add_class("df-card")

display(contenedor_df)